In [ ]:
# import os
# import sys

# current_dir = os.getcwd()
# project_root = os.path.abspath(os.path.join(current_dir,"..","..",".."))
# sys.path.append(project_root)


['C:\\Users\\shuja\\AppData\\Roaming\\uv\\python\\cpython-3.12.11-windows-x86_64-none\\python312.zip', 'C:\\Users\\shuja\\AppData\\Roaming\\uv\\python\\cpython-3.12.11-windows-x86_64-none\\DLLs', 'C:\\Users\\shuja\\AppData\\Roaming\\uv\\python\\cpython-3.12.11-windows-x86_64-none\\Lib', 'C:\\Users\\shuja\\AppData\\Roaming\\uv\\python\\cpython-3.12.11-windows-x86_64-none', 'e:\\Databricks\\Databricks_CICD_Practice\\deb_cicd_prac_project\\.venv_dbc', '', 'e:\\Databricks\\Databricks_CICD_Practice\\deb_cicd_prac_project\\.venv_dbc\\Lib\\site-packages', 'e:\\Databricks\\Databricks_CICD_Practice\\deb_cicd_prac_project\\.venv_dbc\\Lib\\site-packages\\win32', 'e:\\Databricks\\Databricks_CICD_Practice\\deb_cicd_prac_project\\.venv_dbc\\Lib\\site-packages\\win32\\lib', 'e:\\Databricks\\Databricks_CICD_Practice\\deb_cicd_prac_project\\.venv_dbc\\Lib\\site-packages\\pythonwin', 'e:\\Databricks\\Databricks_CICD_Practice\\deb_cicd_prac_project', 'e:\\Databricks\\Databricks_CICD_Practice\\deb_cicd_pr

In [ ]:
from citibike.citibike_utils import get_trip_duration_mins
from utils.datetime_utils import timestamp_to_date_col
from pyspark.sql.functions import create_map, lit

In [3]:
df = spark.read.table(f"citibike_dev.01_bronze.jc_citibike")

In [5]:
df = get_trip_duration_mins(spark, df, "started_at", "ended_at", "trip_duration_mins")

In [6]:
df = timestamp_to_date_col(spark, df, "started_at", "trip_start_date")

In [8]:
df = df.withColumn("metadata", 
              create_map(
                  lit("pipeline_id"), lit("pipeline_id"),
                  lit("run_id"), lit("run_id"),
                  lit("task_id"), lit("task_id"),
                  lit("processed_timestamp"), lit("processed_timestamp")
                  ))

In [10]:
df = df.select(
    "ride_id",
    "trip_start_date",
    "started_at",
    "ended_at",
    "start_station_name",
    "end_station_name",
    "trip_duration_mins",
    "metadata"
    )

In [12]:
df.write.\
    mode("overwrite").\
    option("overwriteSchema", "true").\
    saveAsTable(f"citibike_dev.02_silver.jc_citibike")